In [1]:
import os
import optuna
import pandas as pd
from optuna.trial import TrialState
from optuna.study import StudyDirection

In [2]:
pd.set_option('display.precision', 25)

In [12]:
import os
import pandas as pd
import optuna
from optuna.trial import TrialState

def best_trial(
    db_path: str,
    study_name: str,
    n_trials: int = 50,
    direction: str = 'min'
):
    study = optuna.load_study(
        study_name=study_name,
        storage=f"sqlite:///{db_path}"
    )

    first_n = sorted(study.trials, key=lambda t: t.number)[:n_trials]
    completed = [t for t in first_n if t.state == TrialState.COMPLETE]

    if not completed:
        raise ValueError("No completed trials found in the first N trials.")

    if direction == 'min':
        best = min(completed, key=lambda t: t.value)
    elif direction == 'max':
        best = max(completed, key=lambda t: t.value)
    else:
        raise ValueError("Direction must be 'min' or 'max'")

    return {
        'trial_number': best.number,
        'value': best.value,
        'learning_rate': best.params.get('learning_rate'),
        'weight_decay': best.params.get('weight_decay'),
        'pooling': best.params.get('pooling'),
        'use_numeric': best.params.get('use_numeric'),
        'dropout':best.params.get('dropout')
    }

# Loop and collect results
dbs_path = '/scratch/sas10092/ehr-foundation/models/optuna_dbs/'
dbs = os.listdir(dbs_path)

rows = []
for db in dbs:
    arch = db.split('_')[0]
    if arch == 'big':
        arch = 'big_bird'
    task = db[len(arch)+1:-3]

    result = best_trial(
        db_path=os.path.join(dbs_path, db),
        study_name=arch,
        n_trials=25,
        direction='min'
    )
    row = {
        'arch': arch,
        'task': task,
        **result
    }
    rows.append(row)

# Convert to DataFrame and reorder columns
df = pd.DataFrame(rows)
# df = df[['arch', 'task', 'trial_number', 'value', 'learning_rate', 'weight_decay', 'pooling', 'use_numeric','dropout']]

# Optionally save to CSV


# Show result
df = df.sort_values(['task'])
# df = df[df.arch != 'bert']
df = df.reset_index(drop=True)
# df.to_csv("best_trials_summary.csv", index=False)

In [15]:
df#[df.dropout.isna()==False]#.sort_values('arch')

,arch,task,trial_number,value,learning_rate,weight_decay,pooling,use_numeric,dropout
0,bert,behrt_y_icu_readmit_30,24,0.1479659527540206909179688,0.0004410003173006130702122,NaN,None,None,NaN
1,bert,behrt_y_los_7,12,0.2906840443611145019531250,0.0004937993814347803062798,NaN,None,None,NaN
2,bert,behrt_y_mort,18,0.1813116371631622314453125,0.0004461816873784076953671,NaN,None,None,NaN
3,bert,behrt_y_mort_1yr,12,0.2511317431926727294921875,0.0004912898954059036161698,NaN,None,None,NaN
4,remed,bert-ft_y_icu_readmit_30,5,0.1592701822519302368164062,0.0006827342880425098193761,NaN,None,None,NaN
5,descemb,bert-ft_y_icu_readmit_30,9,0.1541414111852645874023438,0.0002796630109406860018213,NaN,None,None,0.1000000000000000055511151
6,genhpf,bert-ft_y_icu_readmit_30,16,0.1495129615068435668945312,0.0000850304259057302704895,NaN,None,None,0.1000000000000000055511151
7,descemb,bert-ft_y_los_7,19,0.2916699051856994628906250,0.0000732437572907189334568,NaN,None,None,0.1000000000000000055511151
8,genhpf,bert-ft_y_los_7,10,0.2927992939949035644531250,0.0000159097362635984281931,NaN,None,None,0.1000000000000000055511151
9,genhpf,bert-ft_y_mort,18,0.1655942052602767944335938,0.0000995249896829537985610,NaN,None,None,0.1000000000000000055511151


In [11]:
# change this
study = df.iloc[1]
study

arch                     bert
task             behrt_y_mort
trial_number               18
value              0.18131164
learning_rate      0.00044618
weight_decay              NaN
pooling                  None
use_numeric              None
Name: 1, dtype: object

In [12]:
pd.read_csv('best_trials_summary.csv')

,trial_number,value,learning_rate,weight_decay,pooling,use_numeric,task,arch
0,12,0.18271545,0.00004994,0.00158814,cls,False,y_mort,bert
1,17,0.15056728,0.00004911,0.00318868,mean,False,y_icu_readmit_30,longformer
2,12,0.16771893,0.00004991,0.00280652,mean,True,y_mort,modernbert
3,2,0.24706557,0.00004225,0.00216827,cls,False,y_mort_1yr,big_bird
4,11,0.24682316,0.00004858,0.00115370,mean,False,y_mort_1yr,roformer
5,11,0.28047848,0.00004906,0.00111424,cls,False,y_los_7,big_bird
6,16,0.18286677,0.00004929,0.00381852,mean,False,y_mort,longformer
7,2,0.28599778,0.00003635,0.00746513,cls,False,y_los_7,modernbert
8,14,0.17949797,0.00004308,0.00248352,cls,True,y_mort,roberta
9,9,0.29298207,0.00003554,0.00162989,mean,False,y_los_7,longformer


In [16]:
import polars as pl
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

polars.config.Config

In [18]:
pl.read_parquet('./downstream_idx.parquet').filter(pl.col('subject_id') == 10003502)#[:,:30]

subject_id,hadm_id,hosp_admission_time,hosp_discharge_time,icustay_id,icu_admission_time,icu_discharge_time,in_hosp_mort_time,out_mortality_time,n_events_hosp,n_events_icu,shard,hosp_los,hosp_los_hours,hosp_los_days,icu_los,icu_los_hours,icu_los_days,mort_24hr_offset,mort_48hr_offset,y_mort,y_mort_1yr,y_los_7,y_los_15,y_los_30,y_icu_readmit,y_icu_readmit_7,y_icu_readmit_15,y_icu_readmit_30,split,w24_min,w24_max,w48_min,w48_max,wStay_min,wStay_max,w24_start_512,w24_end_512,w24_start_1024,w24_end_1024,w24_start_1536,w24_end_1536,w48_start_512,w48_end_512,w48_start_1024,w48_end_1024,w48_start_1536,w48_end_1536,wStay_start_512,wStay_end_512,wStay_start_1024,wStay_end_1024,wStay_start_1536,wStay_end_1536
i64,i64,datetime[μs],datetime[μs],i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],u32,u32,str,duration[μs],f64,f64,duration[μs],f64,f64,datetime[μs],datetime[μs],i8,i8,i8,i8,i8,i8,i8,i8,i8,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
10003502,29011269,2169-08-26 16:14:00,2169-08-28 15:20:00,35796366,2169-08-26 21:30:32,2169-08-27 22:27:21,null,2169-09-10 00:00:00,893,809,"""9.parquet""",1d 23h 6m,47.1,1.9625,1d 56m 49s,24.946944444444444,1.0394560185185184,2169-08-27 21:30:32,2169-08-28 21:30:32,0,1,0,0,0,0,0,0,0,"""train""",1145,1980,1145,1987,1145,1987,1469,1980,1145,1980,1145,1980,1476,1987,1145,1987,1145,1987,1476,1987,1145,1987,1145,1987
